In [ ]:
import json
import os
import time
from pathlib import Path
from typing import List, Optional, Tuple
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from PIL import Image

# ============================================================
# CONFIG
# ============================================================

# MobileNetV3 is faster than EfficientNet with similar accuracy
MODEL_NAME = "mobilenet_v3_large"

# Larger batch = better GPU utilization (adjust based on VRAM)
BATCH_SIZE = 256

# ImageNet person class is index 281
PERSON_CLASS_IDX = 281

# Threshold for "person" classification (lower = more permissive)
# 0.1-0.3 is typical for pre-filtering
PERSON_CONFIDENCE_THRESHOLD = 0.15

# Parallelism for image decode (CPU+I/O bound)
IMAGE_LOAD_WORKERS = os.cpu_count()
print(f"IMAGE_LOAD_WORKERS: {IMAGE_LOAD_WORKERS}")

# Flush/fsync cadence
CACHE_FSYNC_EVERY_NEW_RECORDS = 10_000
OUT_FSYNC_EVERY_PROMPTS = 1
STATS_FSYNC_EVERY_PROMPTS = 1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using DEVICE: {DEVICE}")

# Cache version (bump if model/threshold/logic changes)
CACHE_VERSION = "mobilenet_v1"

# Enable TF32 on Ampere for throughput
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ============================================================
# LOAD MODEL (ONCE)
# ============================================================

print(f"[Init] Loading {MODEL_NAME}...")
weights = MobileNet_V3_Large_Weights.IMAGENET1K_V2
model = mobilenet_v3_large(weights=weights)
model = model.to(DEVICE)
model.eval()

# Standard ImageNet preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("[Init] Model loaded.")

# ============================================================
# IMAGE PATH RESOLUTION
# ============================================================

def resolve_src(image_root: Path, r: dict) -> Path:
    raw = Path(r["image_path"])
    if raw.is_absolute():
        return raw
    return image_root / raw.parent.parent / f"{r['group_id']}_images" / raw.name

# ============================================================
# VERIFICATION CACHE
# ============================================================

def load_verification_cache(cache_path: Path):
    cache = {}
    if cache_path.exists():
        with open(cache_path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    cache[obj["image_path"]] = obj
                except json.JSONDecodeError:
                    continue
    print(f"[Cache] Loaded {len(cache):,} cached image verifications.")
    return cache

# ============================================================
# PARALLEL IMAGE LOADING & PREPROCESSING
# ============================================================

def _load_and_preprocess(path: Path) -> Optional[torch.Tensor]:
    try:
        with Image.open(path) as im:
            img = im.convert("RGB")
            return preprocess(img)
    except Exception:
        return None

def load_images_parallel(paths: List[Path], max_workers: int) -> Tuple[List[Optional[torch.Tensor]], List[int], torch.Tensor]:
    """
    Returns:
      tensors: list aligned to paths (None if failed)
      valid_indices: indices where tensor != None
      batch_tensor: stacked tensor of valid images [N, 3, 224, 224]
    """
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        tensors = list(ex.map(_load_and_preprocess, paths))

    valid_indices = [i for i, t in enumerate(tensors) if t is not None]
    
    if not valid_indices:
        return tensors, valid_indices, torch.empty(0)
    
    valid_tensors = [tensors[i] for i in valid_indices]
    batch_tensor = torch.stack(valid_tensors)
    
    return tensors, valid_indices, batch_tensor

# ============================================================
# MOBILENET VERIFICATION (BATCHED + OPTIMIZED)
# ============================================================

def verify_batch(image_paths: List[Path]):
    """
    Runs MobileNet classification to detect presence of person.
    
    Returns per-image dicts aligned to image_paths:
      {
        "has_person": bool,
        "person_confidence": float
      }
    """
    if not image_paths:
        return []

    tensors, valid_indices, batch_tensor = load_images_parallel(image_paths, IMAGE_LOAD_WORKERS)

    # Default output: all invalid/unreadable images => rejected
    out = [{"has_person": False, "person_confidence": 0.0} for _ in image_paths]

    if len(valid_indices) == 0:
        return out

    # Run inference
    batch_tensor = batch_tensor.to(DEVICE)
    
    with torch.no_grad():
        logits = model(batch_tensor)
        probs = F.softmax(logits, dim=1)
        person_probs = probs[:, PERSON_CLASS_IDX].cpu().tolist()

    # Populate results for valid images
    for local_idx, global_idx in enumerate(valid_indices):
        confidence = person_probs[local_idx]
        out[global_idx] = {
            "has_person": confidence >= PERSON_CONFIDENCE_THRESHOLD,
            "person_confidence": float(confidence)
        }

    # Help GC
    del tensors, batch_tensor, logits, probs
    
    return out

# ============================================================
# MAIN FILTERING LOOP
# ============================================================

def filter_jsonl(
    input_jsonl: str,
    output_jsonl: str,
    image_root: str
):
    image_root = Path(image_root)
    output_path = Path(output_jsonl)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    stats_path = output_path.with_suffix(".stats.jsonl")
    cache_path = output_path.with_suffix(f".verification_cache.{CACHE_VERSION}.jsonl")

    verification_cache = load_verification_cache(cache_path)

    # Counters to control fsync cadence
    new_cache_records_since_fsync = 0
    prompts_since_out_fsync = 0
    prompts_since_stats_fsync = 0

    try:
        with open(cache_path, "a", encoding="utf-8") as cache_f, \
             open(stats_path, "a", encoding="utf-8") as stats_f, \
             open(input_jsonl, "r", encoding="utf-8") as fin, \
             open(output_jsonl, "a", encoding="utf-8") as fout:

            for line in fin:
                entry = json.loads(line)

                orig_count = len(entry["results"])
                kept_results = []
                results = entry["results"]

                t0 = time.time()

                with tqdm(
                    total=len(results),
                    desc=f"Images [{entry['prompt']}]",
                    unit="img",
                    leave=True,
                    mininterval=0.0
                ) as img_pbar:

                    for i in range(0, len(results), BATCH_SIZE):
                        chunk = results[i:i + BATCH_SIZE]
                        resolved = [resolve_src(image_root, r) for r in chunk]

                        to_verify = []
                        to_verify_indices = []
                        verifications = [{} for _ in resolved]

                        # Cache lookup
                        for idx, path in enumerate(resolved):
                            key = str(path)
                            cached = verification_cache.get(key)
                            if cached is not None:
                                verifications[idx] = cached
                            else:
                                to_verify.append(path)
                                to_verify_indices.append(idx)

                        # Classification only for unseen images
                        if to_verify:
                            new_results = verify_batch(to_verify)

                            for local_idx, global_idx in enumerate(to_verify_indices):
                                v = new_results[local_idx]
                                verifications[global_idx] = v

                                record = {
                                    "image_path": str(resolved[global_idx]),
                                    "has_person": bool(v.get("has_person", False)),
                                    "person_confidence": float(v.get("person_confidence", 0.0))
                                }

                                verification_cache[record["image_path"]] = record
                                cache_f.write(json.dumps(record) + "\n")
                                new_cache_records_since_fsync += 1

                            # Flush/fsync cache periodically
                            if new_cache_records_since_fsync >= CACHE_FSYNC_EVERY_NEW_RECORDS:
                                cache_f.flush()
                                os.fsync(cache_f.fileno())
                                new_cache_records_since_fsync = 0

                        # Apply acceptance logic
                        for r, v in zip(chunk, verifications):
                            if v.get("has_person", False):
                                r.update(v)
                                kept_results.append(r)
                        
                        img_pbar.update(len(chunk))

                kept_count = len(kept_results)
                retention = kept_count / orig_count if orig_count > 0 else 0.0
                elapsed = time.time() - t0

                tqdm.write(
                    f"[{entry['prompt']}] "
                    f"original={orig_count} kept={kept_count} "
                    f"retention={retention:.2%} "
                    f"time={elapsed/60:.2f}min"
                )

                # Write filtered output
                if kept_results:
                    fout.write(json.dumps({
                        "prompt": entry["prompt"],
                        "original_count": orig_count,
                        "kept_count": kept_count,
                        "results": kept_results
                    }) + "\n")
                    prompts_since_out_fsync += 1

                    if prompts_since_out_fsync >= OUT_FSYNC_EVERY_PROMPTS:
                        fout.flush()
                        os.fsync(fout.fileno())
                        prompts_since_out_fsync = 0

                # Write stats
                stats_f.write(json.dumps({
                    "prompt": entry["prompt"],
                    "original_count": orig_count,
                    "kept_count": kept_count,
                    "retention": retention,
                    "seconds": elapsed
                }) + "\n")
                prompts_since_stats_fsync += 1

                if prompts_since_stats_fsync >= STATS_FSYNC_EVERY_PROMPTS:
                    stats_f.flush()
                    os.fsync(stats_f.fileno())
                    prompts_since_stats_fsync = 0

            # Final flushes
            cache_f.flush()
            os.fsync(cache_f.fileno())
            fout.flush()
            os.fsync(fout.fileno())
            stats_f.flush()
            os.fsync(stats_f.fileno())

    except KeyboardInterrupt:
        print("\nInterrupted — progress safely flushed.")
        try:
            cache_f.flush()
            os.fsync(cache_f.fileno())
        except Exception:
            pass
        try:
            fout.flush()
            os.fsync(fout.fileno())
        except Exception:
            pass
        try:
            stats_f.flush()
            os.fsync(stats_f.fileno())
        except Exception:
            pass
        raise

# ============================================================
# ENTRY POINT
# ============================================================

if __name__ == "__main__":
    filter_jsonl(
        input_jsonl=r"G:\Thesis\ImageRetrieval\Professions_20k\retrieval_results_batchsize_10.jsonl",
        output_jsonl=r"G:\Thesis\ImageRetrieval\testing\retrieval_results_people_verified.jsonl",
        image_root=r"G:\Thesis"
    )

IMAGE_LOAD_WORKERS: 16
Using DEVICE: cuda
[Init] Loading mobilenet_v3_large...


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to C:\Users\User/.cache\torch\hub\checkpoints\mobilenet_v3_large-5c1a4163.pth
100%|██████████| 21.1M/21.1M [00:01<00:00, 19.4MB/s]


[Init] Model loaded.
[Cache] Loaded 0 cached image verifications.


Images [Male Accountant]:  74%|███████▎  | 1469184/1996559 [2:02:21<3:20:09, 43.91img/s] 